# Raw Data Download

Download raw experimental data files from the aimdl/datafiles API with configurable data types.

This notebook demonstrates how to:
- Query files by data type using the aimdl/datafiles endpoint
- Download multiple files in parallel
- Save to disk with metadata

In [ ]:
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import get_girder_client, download_item_to_disk

## Configuration

In [ ]:
from dotenv import load_dotenv

load_dotenv('..')

# Configure data type to download
# Change this to fetch different data types (e.g., "pdv_raw", "xrd_raw", etc.)
DATA_TYPE = "pdv_raw"

# Output directory for downloaded files
OUTPUT_DIR = f"./raw_data_{DATA_TYPE}"

## Download Files

In [ ]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    results = []
    limit = 50
    offset = 0
    
    with ThreadPoolExecutor(max_workers=8) as executor:
        while True:
            batch = gc.get(
                "aimdl/datafiles",
                parameters={
                    "limit": limit,
                    "offset": offset,
                    "dataType": DATA_TYPE,
                },
            )
            if not batch:
                break
            
            futures = [
                executor.submit(download_item_to_disk, gc, item, OUTPUT_DIR)
                for item in batch
            ]
            
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                    print(f"Downloaded: {result['name']} ({result['size'] / 1024:.2f} KB)")
                except Exception as e:
                    print(f"Error: {e}")
            
            if len(batch) < limit:
                break
            offset += limit

print(f"\\nTotal files downloaded: {len(results)}")
print(f"Total size: {sum(r['size'] for r in results) / (1024**2):.2f} MB")

## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"Downloaded {len(df)} files of type '{DATA_TYPE}'")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\\nSample files:")
print(df[["name", "size"]].head(10))